# Session 1 Lecture: AI Data Platform Reference Architectures & Constraints


## The "Data for AI" problem

Traditional data platforms are optimised for BI and reporting. AI systems impose fundamentally different requirements:

| Requirement | BI / Reporting | AI / Agents |
|---|---|---|
| Data types | Structured tables | Text, PDFs, images, audio, video + structured |
| Freshness | Batch (daily/weekly) | Near-real-time or on-demand |
| Governance | Row/column security | Fine-grained document + embedding provenance |
| Quality | Completeness, accuracy | Semantic accuracy, context fidelity |
| Discoverability | Schema browser | Semantic search over data AND metadata |

**Key insight:** AI systems are only as good as the data they're grounded in. A poorly governed data layer produces hallucinating agents.

## The Medallion Architecture for multi-modal data

The medallion pattern (Bronze → Silver → Gold) works for unstructured data — but the transformation steps change:

```
Raw PDFs / documents              ← Binary files in Volumes (Bronze)

         ↓  ai_parse_document()

Parsed text + document structure  ← Extracted text, tables, sections (Silver)

         ↓  ai_extract() / ai_classify()

Structured insights + embeddings  ← Named entities, metrics, categories (Gold)

         ↓

Vector index + Genie + Agents     ← Consumption layer
```

**What changes at each layer:**
- **Bronze:** Raw fidelity — store exactly what was ingested, with provenance (source path, timestamp)
- **Silver:** Semantic fidelity — AI functions parse and extract; data quality expectations applied
- **Gold:** Business fidelity — classification, aggregation, enrichment; the layer consultants query

## Databricks platform map for AI data platforms

```
┌─────────────────────────────────────────────────────────────────────┐
│                         Unity Catalog                               │
│  Governance · Lineage · Access Control · Discoverability            │
├──────────────┬────────────────────────────────────┬─────────────────┤
│  LakeFlow    │  Spark Declarative Pipelines       │  LakeFlow       │
│  Connect     │  (automated medallion)             │  Jobs           │
│  (ingestion) │                                    │  (orchestration)│
├──────────────┴────────────────────────────────────┴─────────────────┤
│         Delta Lake + Volumes (storage)                              │
├─────────────────────────────────────────────────────────────────────┤
│   AI Functions: ai_parse_document · ai_extract · ai_classify        │
├─────────────────────────────────────────────────────────────────────┤
│     Agent Bricks · Vector Search · Genie · Model Serving            │
└─────────────────────────────────────────────────────────────────────┘
```

**Products used in this workshop:**
- **Unity Catalog** — governance, lineage, access control, tagging
- **Volumes** — managed storage for binary files (PDFs)
- **Lakeflow Spark Declarative Pipelines (LSDP)** — automated bronze→silver→gold pipeline
- **AI Functions** — `ai_parse_document`, `ai_extract`, `ai_classify` — SQL-native document AI
- **LakeFlow Jobs** — workflow orchestration (pipeline → validate → notify)
- **Declarative Automation Bundles (DABs)** — infrastructure as code for the above

## Key design constraints

-  **Latency:** `ai_parse_document` and `ai_extract` are compute-intensive. Design batch pipelines, not row-at-a-time triggers. Expect 1–5 seconds per document at scale.

-  **Cost:** AI function calls consume Foundation Model API tokens. Budget accordingly — parsing 130 PDFs at 50 pages each costs approximately [FMaaS pricing] in token usage.

-  **Freshness:** New SEC filings drop quarterly. Use scheduled pipelines (not continuous) to process new documents on a cadence aligned with filing dates.

-  **Trust:** Data quality expectations in Spark Declarative Pipelines create an audit trail. `expect_or_drop` ensures downstream tables never contain garbage — the pipeline surfaces the failure rather than silently propagating bad data.

-  **Discoverability:** Unity Catalog tags and lineage make the data platform self-documenting. An analyst can discover `company_ai_investment_summary` and trace it all the way back to the source PDF.

## Today's scenario: Financial Intelligence Platform

**Business problem:** Advisory analysts spend hours reading 10-Ks and earnings call transcripts to understand a company's AI investment posture. Can we automate the extraction and make it queryable?

**What we're building:**
- Ingest SEC filings and earnings materials for Apple, Amazon, Microsoft, NVIDIA, Meta
- Parse and extract key financial metrics and AI investment signals
- Classify management sentiment and AI investment category per document
- Serve a queryable gold layer that answers: *"How are these companies investing in AI infrastructure, and what's the sentiment trend?"*

**This is a typical pattern you would implement for a client.**

## Discussion questions (for instructor to prompt)

1. Which of your current clients have a data platform that could support AI agents today? What's missing?
2. Where in the medallion do you think the highest-risk quality failures occur?
3. How would you govern access to sensitive financial documents in a client environment?